# N053 · 队列、双端队列与循环缓冲区

**目标：** 实现FIFO并解释容器操作成本。

**先修：** N008, N014, N051。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 环形数组；head/tail/size；满空判定；双端操作；两栈队列。

## 从问题到算法

循环缓冲区用固定数组和模运算复用已释放位置，避免每次出队整体搬移。必须额外记录size或留空槽来区分“头尾相同”的空与满；这里选择size。双端队列允许从两端插删，普通队列只是限制其中两种操作。容量必须为正。

## 最小实现

**本章接口：** `CircularQueue`、`CircularDeque`、`TwoStackQueue`

In [1]:
class CircularDeque:
    def __init__(self,k):
        if k<1: raise ValueError('positive capacity required')
        self.data=[None]*k; self.head=0; self.size=0
    def isEmpty(self): return self.size==0
    def isFull(self): return self.size==len(self.data)
    def insertFront(self,value):
        if self.isFull(): return False
        self.head=(self.head-1)%len(self.data)
        self.data[self.head]=value; self.size+=1; return True
    def insertLast(self,value):
        if self.isFull(): return False
        self.data[(self.head+self.size)%len(self.data)]=value
        self.size+=1; return True
    def deleteFront(self):
        if self.isEmpty(): return False
        self.data[self.head]=None; self.head=(self.head+1)%len(self.data)
        self.size-=1; return True
    def deleteLast(self):
        if self.isEmpty(): return False
        self.size-=1; self.data[(self.head+self.size)%len(self.data)]=None; return True
    def getFront(self): return -1 if self.isEmpty() else self.data[self.head]
    def getRear(self): return -1 if self.isEmpty() else self.data[(self.head+self.size-1)%len(self.data)]

class CircularQueue(CircularDeque):
    enQueue=CircularDeque.insertLast
    deQueue=CircularDeque.deleteFront
    Front=CircularDeque.getFront
    Rear=CircularDeque.getRear

class TwoStackQueue:
    def __init__(self): self.incoming=[]; self.outgoing=[]
    def push(self,x): self.incoming.append(x)
    def pop(self):
        if not self.outgoing:
            while self.incoming: self.outgoing.append(self.incoming.pop())
        if not self.outgoing: raise IndexError('empty queue')
        return self.outgoing.pop()

## 正确性、前提与复杂度

第i个逻辑元素总位于(head+i) mod capacity。四种操作只改变端点位置与size，按这个映射核对即可证明顺序不变。

**代价：** 循环队列各操作最坏O(1)，空间O(capacity)；双栈队列单次最坏O(n)、均摊O(1)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

q=CircularQueue(3); rows=[]
for op,value in [('put',1),('put',2),('get',None),('put',3),('put',4),('get',None)]:
    if op=='put': q.enQueue(value)
    else: q.deQueue()
    rows.append((op,value,q.head,q.size,q.data[:]))
show_table(['操作','值','head','size','物理数组'],rows)

操作,值,head,size,物理数组
put,1,0,1,"[1, None, None]"
put,2,0,2,"[1, 2, None]"
get,None,1,1,"[None, 2, None]"
put,3,1,2,"[None, 2, 3]"
put,4,1,3,"[4, 2, 3]"
get,None,2,2,"[4, None, 3]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from collections import deque
from random import Random
q=CircularQueue(1); assert q.enQueue(7) and not q.enQueue(8)
assert q.Front()==q.Rear()==7 and q.deQueue() and not q.deQueue()
rng=Random(53); q=CircularDeque(5); ref=deque()
for _ in range(1000):
    op=rng.randrange(4); x=rng.randrange(20)
    if op<2:
        expected=len(ref)<5
        actual=q.insertFront(x) if op==0 else q.insertLast(x)
        if expected:
            if op==0: ref.appendleft(x)
            else: ref.append(x)
    else:
        expected=bool(ref); actual=q.deleteFront() if op==2 else q.deleteLast()
        if expected:
            if op==2: ref.popleft()
            else: ref.pop()
    assert actual==expected
    assert q.size==len(ref) and q.getFront()==(ref[0] if ref else -1) and q.getRear()==(ref[-1] if ref else -1)
t=TwoStackQueue()
for x in range(5): t.push(x)
assert [t.pop() for _ in range(5)]==list(range(5))
print('N053: 所有本章断言通过')

N053: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 不用额外size区分满与空并说明容量变化。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较deque与list.pop(0)。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 622. Design Circular Queue（canonical）
- 641. Design Circular Deque（canonical）
- 232. Implement Queue using Stacks（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。